In [4]:
import pandas as pd
import numpy as np
import datetime as dt
import time

# DTW package: 
from dtaidistance import dtw

import plotly.express as px
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
data = pd.read_csv("../data/preprocessed_data.csv")
data.head()

,Unnamed: 0,timestamp,load_Rotary_C5_temp,execution,program_name,nsequence,cr,index
0,46,2024-03-11 14:01:19.408233,0.014493,0,O0005(5303-005-C),N10,CR-1,0
1,49,2024-03-11 14:01:29.432814,0.014493,0,O0005(5303-005-C),N10,CR-1,1
2,63,2024-03-11 14:01:37.451878,0.000000,0,O0005(5303-005-C),N10,CR-1,2
3,64,2024-03-11 14:01:44.468672,0.000000,0,O0005(5303-005-C),N10,CR-1,3
4,78,2024-03-11 14:01:52.489886,0.000000,0,O0005(5303-005-C),N10,CR-1,4


In [56]:
fig = px.line(data, x="index", y="load_Rotary_C5_temp", color='cr')
fig.update_layout(title_text="Load Vs Index")
fig.show()

In [3]:
print(dtw)

<module 'dtaidistance.dtw' from 'c:\\Users\\kashika.p\\AppData\\Local\\miniconda3\\envs\\baseline_model\\Lib\\site-packages\\dtaidistance\\dtw.py'>


In [14]:
def prepare_sequences_for_dtw(data): 
    start_time = time.time()

    print("===== STEP 1: Preparing Sequences =====")
    grouped = data.groupby('cr')['load_Rotary_C5_temp'].apply(list)
    cr_load_curves = grouped.tolist()

    elapsed_time = time.time() - start_time 
    print("Time taken in STEP 1: ", elapsed_time)

    return cr_load_curves


In [15]:
cr_load_curves = prepare_sequences_for_dtw(data)

===== STEP 1: Preparing Sequences =====
Time taken in STEP 1:  0.005313873291015625


### basic cost compute: 

In [80]:
def euclidean_distance(x, y):
    return abs(x - y)

def compute_distance_matrix(unique_crs, curves):
    """
    Incrementally computes DTW pairwise distance matrix.
    
    Returns:
        distance_df: pandas DataFrame (pairwise distances)
    """
    
    start_time = time.time()
    print("===== STEP 2: Computing DTW distance matrix incrementally =====")
    
    n = len(curves)
    distance_matrix = np.zeros((n, n))

    for i in range(n):
        for j in range(i, n):  # leverage symmetry
            if i == j:
                dist = 0
            else:
                dist = dtw.distance(curves[i], curves[j])
                
            
            distance_matrix[i, j] = dist
            distance_matrix[j, i] = dist  # symmetric fill
            print(f"CR {i+1}-{j+1}: {dist}")

    elapsed_time = time.time() - start_time 
    distance_df = pd.DataFrame(distance_matrix, index=unique_crs, columns=unique_crs)
    
    print("Time taken in STEP 2: ", elapsed_time)
    
    return distance_df

In [81]:
unique_crs = data['cr'].unique()
print(type(unique_crs))

distance_df = compute_distance_matrix(unique_crs, cr_load_curves)

<class 'pandas.arrays.StringArray'>
===== STEP 2: Computing DTW distance matrix incrementally =====
CR 1-1: 0
CR 1-2: 0.67930676204394
CR 1-3: 1.648483037310853
CR 1-4: 2.18220640286594
CR 1-5: 1.8773570915607312
CR 1-6: 1.8085771979048477
CR 1-7: 2.0364252835072834
CR 1-8: 1.444630237029228
CR 1-9: 1.933085615227426
CR 1-10: 2.1878778516644464
CR 1-11: 1.8775249050162064
CR 1-12: 1.621375799530086
CR 1-13: 2.077676630429849
CR 1-14: 1.4592410983352873
CR 1-15: 1.944190879737278
CR 1-16: 1.780958349990935
CR 2-2: 0
CR 2-3: 1.7839631946004757
CR 2-4: 2.284069006079424
CR 2-5: 1.9993697809832536
CR 2-6: 1.7011554445937231
CR 2-7: 2.083380591776535
CR 2-8: 1.499632385114825
CR 2-9: 2.049841055103228
CR 2-10: 2.2894881379092964
CR 2-11: 1.9995273543594674
CR 2-12: 1.753563299971499
CR 2-13: 2.1237200699521988
CR 2-14: 1.5137124402662592
CR 2-15: 1.9262826914426165
CR 2-16: 1.9011487920935173
CR 3-3: 0
CR 3-4: 1.4776213450405575
CR 3-5: 1.236648926376335
CR 3-6: 1.4049751070605374
CR 3-7: 1

In [122]:
def detect_outliers(distance_df, iqr_multiplier=0.05):
    """
    Detects outliers based on mean DTW distance using IQR method.
    
    Returns:
        valid_rounds: list
        outlier_rounds: list
        mean_distances: pd.Series
    """
    print("===== STEP 3: Detecting outliers =====")
    
    start_time = time.time()
    # Mean distance excluding self-distance
    mean_distances = distance_df.apply(
        lambda row: row.drop(labels=row.name).mean(), axis=1
    )

    # IQR calculation
    Q1 = mean_distances.quantile(0.25)
    Q3 = mean_distances.quantile(0.75)
    IQR = Q3 - Q1
    upper_bound = Q3 + iqr_multiplier * IQR

    outlier_rounds = mean_distances[mean_distances > upper_bound].index.tolist()
    valid_rounds = mean_distances[mean_distances <= upper_bound].index.tolist()

    elapsed_time = time.time() - start_time 
    print("Time taken in STEP 3: ", elapsed_time)

    print("Outlier CRs: ", outlier_rounds)
    print("Valid CRs: ", valid_rounds)
    print("Threshold:", upper_bound)
    print(f'Mean distances per CR (index i = CR-(i+1)): {[round(m, 5) for m in mean_distances]}')

    return valid_rounds, outlier_rounds, mean_distances

In [123]:
_, outlier_rounds, _ = detect_outliers(distance_df, 0.05)

===== STEP 3: Detecting outliers =====
Time taken in STEP 3:  0.015588045120239258
Outlier CRs:  ['CR-1', 'CR-2', 'CR-4', 'CR-11']
Valid CRs:  ['CR-3', 'CR-5', 'CR-6', 'CR-7', 'CR-8', 'CR-9', 'CR-10', 'CR-12', 'CR-13', 'CR-14', 'CR-15', 'CR-16']
Threshold: 2.4850423958526893
Mean distances per CR (index i = CR-(i+1)): [2.66676, 2.86545, 2.14369, 3.0093, 2.44996, 2.03876, 1.98751, 1.8908, 2.13976, 2.35729, 2.50112, 2.03876, 2.02676, 1.83893, 2.09287, 1.93753]


### Cost compute using Normliazation: 

In [75]:
# With Normalization of costs based on lengths 
def custom_dtw_normalization(dist, curve_j):
    """
    Custom normalization: divide DTW cost by length of curve_j.
    
    This mimics your existing logic:
    cost[i][j] / len(curve_j)
    
    Args:
        dist: raw DTW distance
        curve_j: reference/query sequence (j)
    
    Returns:
        normalized distance
    """
    size_j = len(curve_j)
    
    if size_j == 0:
        return 0  # safe guard
    
    return round(dist / size_j, 5)

In [76]:
def compute_distance_matrix_with_norm(unique_crs, curves, use_custom_norm=False):
    """
    Computes DTW distance matrix with optional custom normalization.

    Args:
        unique_crs: list of CR/round identifiers
        curves: list of sequences
        use_custom_norm: bool → apply your normalization if True

    Returns:
        distance_df
    """
    
    print(f"Computing DTW matrix | Custom normalization: {use_custom_norm}")
    start_time = time.time()

    n = len(curves)
    distance_matrix = np.zeros((n, n))

    for i in range(n):
        for j in range(i, n):

            if i == j:
                dist = 0
            else:
                raw_dist = dtw.distance(curves[i], curves[j])

                if use_custom_norm:
                    # Apply normalization (based on curve_j)
                    dist = custom_dtw_normalization(raw_dist, curves[j])

                else:
                    dist = raw_dist

            # Maintain your symmetric assignment
            distance_matrix[i, j] = dist
            distance_matrix[j, i] = dist
            print(f"CR {i+1}-{j+1}: {dist}")

    elapsed_time = time.time() - start_time
    print("DTW Cost Calculation (with Normalization): ", elapsed_time)

    return pd.DataFrame(distance_matrix, index=unique_crs, columns=unique_crs)

In [77]:
unique_crs = data['cr'].unique()
print(type(unique_crs))

distance_df = compute_distance_matrix_with_norm(unique_crs, cr_load_curves, use_custom_norm=True)

<class 'pandas.arrays.StringArray'>
Computing DTW matrix | Custom normalization: True
CR 1-1: 0
CR 1-2: 0.00087
CR 1-3: 0.00252
CR 1-4: 0.00334
CR 1-5: 0.00252
CR 1-6: 0.00337
CR 1-7: 0.00389
CR 1-8: 0.00273
CR 1-9: 0.00291
CR 1-10: 0.00329
CR 1-11: 0.00254
CR 1-12: 0.00303
CR 1-13: 0.00391
CR 1-14: 0.00275
CR 1-15: 0.00361
CR 1-16: 0.00336
CR 2-2: 0
CR 2-3: 0.00272
CR 2-4: 0.00349
CR 2-5: 0.00269
CR 2-6: 0.00317
CR 2-7: 0.00398
CR 2-8: 0.00283
CR 2-9: 0.00308
CR 2-10: 0.00345
CR 2-11: 0.0027
CR 2-12: 0.00328
CR 2-13: 0.004
CR 2-14: 0.00285
CR 2-15: 0.00358
CR 2-16: 0.00359
CR 3-3: 0
CR 3-4: 0.00226
CR 3-5: 0.00166
CR 3-6: 0.00262
CR 3-7: 0.00226
CR 3-8: 0.00186
CR 3-9: 0.00157
CR 3-10: 0.00226
CR 3-11: 0.00174
CR 3-12: 0.00212
CR 3-13: 0.00224
CR 3-14: 0.00193
CR 3-15: 0.00222
CR 3-16: 0.00203
CR 4-4: 0
CR 4-5: 0.00207
CR 4-6: 0.00298
CR 4-7: 0.00167
CR 4-8: 0.00269
CR 4-9: 0.00174
CR 4-10: 0.00039
CR 4-11: 0.00215
CR 4-12: 0.00237
CR 4-13: 0.00163
CR 4-14: 0.00272
CR 4-15: 0.00188
CR

In [70]:
_, outlier_rounds, _ = detect_outliers(distance_df, 0.001) 

===== STEP 3: Detecting outliers =====
Time taken in STEP 3:  0.009029865264892578
Outlier CRs:  ['CR-1', 'CR-2', 'CR-4', 'CR-6']
Valid CRs:  ['CR-3', 'CR-5', 'CR-7', 'CR-8', 'CR-9', 'CR-10', 'CR-11', 'CR-12', 'CR-13', 'CR-14', 'CR-15', 'CR-16']
Threshold: 0.002235526166666667
Mean distances per CR (index i = CR-(i+1)): [0.00298, 0.00309, 0.00213, 0.00227, 0.00199, 0.00231, 0.00195, 0.00193, 0.00221, 0.00222, 0.00187, 0.00206, 0.00209, 0.00208, 0.00214, 0.00214]


### Segment-wise optimization: 

In [110]:
# Segment-wise optimization: 
def extract_segments(curve, num_segments = 6, segment_indices=[0, 3, 5]):
        L = len(curve)
        seg_len = max(1, L // num_segments)  # avoid zero division

        segments = []
        for idx in segment_indices:
            start = idx * seg_len
            end = min((idx + 1) * seg_len, L)
            seg = curve[start:end]

            # Skip empty segments
            if len(seg) > 1:
                segments.append(seg)

        return segments

def compute_distance_matrix_in_segments(
    rounds,
    curves,
    num_segments=6,
    segment_indices=[0, 3, 5]
):
    """
    Computes DTW distance matrix using segment-wise optimization.

    Strategy:
    - Split each curve into segments
    - Compute DTW only on selected segments
    - Average segment-wise DTW costs

    Returns:
        distance_df (pd.DataFrame)
    """
    start_time = time.time()
    print("Computing segment-wise DTW distance matrix...")

    # -------------------------------
    # Step 1: Precompute segments
    # -------------------------------
    segmented_curves = [extract_segments(c, num_segments, segment_indices) for c in curves]

    # -------------------------------
    # Step 2: Distance computation
    # -------------------------------
    N = len(segmented_curves)
    distance_matrix = np.zeros((N, N))

    for i in range(N):
        for j in range(i, N):

            if i == j:
                dist = 0

            else:
                segs_i = segmented_curves[i]
                segs_j = segmented_curves[j]

                dist_sum = 0
                valid_seg_count = 0

                for seg_i, seg_j in zip(segs_i, segs_j):
                    dist_ij = dtw.distance(seg_i, seg_j)

                    dist_sum += dist_ij
                    valid_seg_count += 1

                # Avoid division by zero
                if valid_seg_count == 0:
                    dist = 0
                else:
                    dist = dist_sum 

            distance_matrix[i, j] = dist
            distance_matrix[j, i] = dist
            print(f"CR {i+1}-{j+1}: {dist}")
    
    elapsed_time = time.time() - start_time 
    print("Time taken: ", elapsed_time)

    return pd.DataFrame(distance_matrix, index=rounds, columns=rounds)

In [111]:
unique_crs = data['cr'].unique()
print(type(unique_crs))

distance_df = compute_distance_matrix_in_segments(unique_crs, cr_load_curves)

<class 'pandas.arrays.StringArray'>
Computing segment-wise DTW distance matrix...
CR 1-1: 0
CR 1-2: 1.0345764792936145
CR 1-3: 2.2615188461634115
CR 1-4: 3.1843641390428963
CR 1-5: 2.7992931525621936
CR 1-6: 2.8521376907986777
CR 1-7: 3.054861341066626
CR 1-8: 2.456720129153527
CR 1-9: 2.3397282726454884
CR 1-10: 2.934988490545637
CR 1-11: 2.802684564840665
CR 1-12: 2.8521376907986777
CR 1-13: 3.0701229430893084
CR 1-14: 2.4181497824027844
CR 1-15: 3.1635848504306594
CR 1-16: 2.7765432670324053
CR 2-2: 0
CR 2-3: 2.535497761776413
CR 2-4: 3.393849419576852
CR 2-5: 3.0578977340899183
CR 2-6: 3.0984929672650514
CR 2-7: 3.196959683309901
CR 2-8: 2.58857711290294
CR 2-9: 2.597141794894748
CR 2-10: 3.148219210679599
CR 2-11: 3.060146061348306
CR 2-12: 3.0984929672650514
CR 2-13: 3.236260785382058
CR 2-14: 2.586796443652352
CR 2-15: 3.3660709840637293
CR 2-16: 2.9828200082898704
CR 3-3: 0
CR 3-4: 1.9658191234396156
CR 3-5: 1.8452369604637742
CR 3-6: 2.568149155870126
CR 3-7: 2.659187182052456

In [115]:
_, outlier_rounds, _ = detect_outliers(distance_df, 0.05) 

===== STEP 3: Detecting outliers =====
Time taken in STEP 3:  0.010088682174682617
Outlier CRs:  ['CR-1', 'CR-2', 'CR-4', 'CR-11']
Valid CRs:  ['CR-3', 'CR-5', 'CR-6', 'CR-7', 'CR-8', 'CR-9', 'CR-10', 'CR-12', 'CR-13', 'CR-14', 'CR-15', 'CR-16']
Threshold: 2.4850423958526893
Mean distances per CR (index i = CR-(i+1)): [2.66676, 2.86545, 2.14369, 3.0093, 2.44996, 2.03876, 1.98751, 1.8908, 2.13976, 2.35729, 2.50112, 2.03876, 2.02676, 1.83893, 2.09287, 1.93753]


### Custom distance function: 

In [128]:
def compute_distance_matrix(unique_crs, curves):
    """
    Incrementally computes DTW pairwise distance matrix.
    
    Returns:
        distance_df: pandas DataFrame (pairwise distances)
    """
    
    start_time = time.time()
    print("===== STEP 2: Computing DTW distance matrix incrementally =====")
    
    n = len(curves)
    distance_matrix = np.zeros((n, n))

    for i in range(n):
        for j in range(i, n):  # leverage symmetry
            if i == j:
                dist = 0
            else:
                dist = dtw.distance(curves[i], curves[j], inner_dist='euclidean')
                
            
            distance_matrix[i, j] = dist
            distance_matrix[j, i] = dist  # symmetric fill
            print(f"CR {i+1}-{j+1}: {dist}")

    elapsed_time = time.time() - start_time 
    distance_df = pd.DataFrame(distance_matrix, index=unique_crs, columns=unique_crs)
    
    print("Time taken in custom dist. func: ", elapsed_time)
    
    return distance_df

In [129]:
unique_crs = data['cr'].unique()
print(type(unique_crs))

distance_df = compute_distance_matrix(unique_crs, cr_load_curves)

<class 'pandas.arrays.StringArray'>
===== STEP 2: Computing DTW distance matrix incrementally =====
CR 1-1: 0
CR 1-2: 1.0579710144927537
CR 1-3: 33.60869565217377
CR 1-4: 40.71014492753601
CR 1-5: 37.42028985507228
CR 1-6: 32.78260869565213
CR 1-7: 38.913043478260796
CR 1-8: 28.72463768115924
CR 1-9: 35.98550724637667
CR 1-10: 41.07246376811572
CR 1-11: 37.434782608695464
CR 1-12: 32.20289855072459
CR 1-13: 40.20289855072455
CR 1-14: 28.971014492753447
CR 1-15: 38.27536231884048
CR 1-16: 34.75362318840567
CR 2-2: 0
CR 2-3: 34.507246376811445
CR 2-4: 41.17391304347809
CR 2-5: 38.42028985507225
CR 2-6: 33.60869565217386
CR 2-7: 38.88405797101434
CR 2-8: 29.56521739130415
CR 2-9: 36.88405797101434
CR 2-10: 41.52173913043459
CR 2-11: 38.434782608695436
CR 2-12: 33.15942028985502
CR 2-13: 39.71014492753608
CR 2-14: 29.811594202898352
CR 2-15: 39.11594202898539
CR 2-16: 35.666666666666536
CR 3-3: 0
CR 3-4: 15.202898550724619
CR 3-5: 16.71014492753611
CR 3-6: 13.565217391304266
CR 3-7: 13.710

In [130]:
_, outlier_rounds, _ = detect_outliers(distance_df, 0.05)

===== STEP 3: Detecting outliers =====
Time taken in STEP 3:  0.010190725326538086
Outlier CRs:  ['CR-1', 'CR-2', 'CR-4', 'CR-10']
Valid CRs:  ['CR-3', 'CR-5', 'CR-6', 'CR-7', 'CR-8', 'CR-9', 'CR-11', 'CR-12', 'CR-13', 'CR-14', 'CR-15', 'CR-16']
Threshold: 17.733103864734193
Mean distances per CR (index i = CR-(i+1)): [33.4744, 34.10145, 16.14879, 17.96425, 17.457, 15.28599, 14.69469, 14.97488, 17.27729, 18.457, 17.49372, 14.60097, 15.16618, 15.17971, 16.02512, 15.43188]


### Custom dist + Norm: 

In [131]:
def compute_distance_matrix_with_norm_2(unique_crs, curves, use_custom_norm=True):
    """
    Computes DTW distance matrix with optional custom normalization.

    Args:
        unique_crs: list of CR/round identifiers
        curves: list of sequences
        use_custom_norm: bool → apply your normalization if True

    Returns:
        distance_df
    """
    
    print(f"Computing DTW matrix | Custom normalization: {use_custom_norm}")
    start_time = time.time()

    n = len(curves)
    distance_matrix = np.zeros((n, n))

    for i in range(n):
        for j in range(i, n):

            if i == j:
                dist = 0
            else:
                raw_dist = dtw.distance(curves[i], curves[j], inner_dist='euclidean')

                if use_custom_norm:
                    # Apply normalization (based on curve_j)
                    dist = custom_dtw_normalization(raw_dist, curves[j])

                else:
                    dist = raw_dist

            # Maintain your symmetric assignment
            distance_matrix[i, j] = dist
            distance_matrix[j, i] = dist
            print(f"CR {i+1}-{j+1}: {dist}")

    elapsed_time = time.time() - start_time
    print("DTW Cost Calculation (with Normalization): ", elapsed_time)

    return pd.DataFrame(distance_matrix, index=unique_crs, columns=unique_crs)

In [132]:
unique_crs = data['cr'].unique()
print(type(unique_crs))

distance_df = compute_distance_matrix_with_norm_2(unique_crs, cr_load_curves)

<class 'pandas.arrays.StringArray'>
Computing DTW matrix | Custom normalization: True
CR 1-1: 0
CR 1-2: 0.00136
CR 1-3: 0.05131
CR 1-4: 0.06225
CR 1-5: 0.0503
CR 1-6: 0.06105
CR 1-7: 0.0744
CR 1-8: 0.0543
CR 1-9: 0.05411
CR 1-10: 0.06186
CR 1-11: 0.05059
CR 1-12: 0.06019
CR 1-13: 0.07571
CR 1-14: 0.05456
CR 1-15: 0.07114
CR 1-16: 0.06557
CR 2-2: 0
CR 2-3: 0.05268
CR 2-4: 0.06296
CR 2-5: 0.05164
CR 2-6: 0.06259
CR 2-7: 0.07435
CR 2-8: 0.05589
CR 2-9: 0.05546
CR 2-10: 0.06253
CR 2-11: 0.05194
CR 2-12: 0.06198
CR 2-13: 0.07478
CR 2-14: 0.05614
CR 2-15: 0.07271
CR 2-16: 0.0673
CR 3-3: 0
CR 3-4: 0.02325
CR 3-5: 0.02246
CR 3-6: 0.02526
CR 3-7: 0.02621
CR 3-8: 0.02293
CR 3-9: 0.00399
CR 3-10: 0.02364
CR 3-11: 0.02291
CR 3-12: 0.02427
CR 3-13: 0.02694
CR 3-14: 0.02347
CR 3-15: 0.02459
CR 3-16: 0.02737
CR 4-4: 0
CR 4-5: 0.02447
CR 4-6: 0.03257
CR 4-7: 0.0215
CR 4-8: 0.0329
CR 4-9: 0.02197
CR 4-10: 0.0007
CR 4-11: 0.0254
CR 4-12: 0.03064
CR 4-13: 0.02118
CR 4-14: 0.03338
CR 4-15: 0.02416
CR 4-16

In [133]:
_, outlier_rounds, _ = detect_outliers(distance_df, 0.05)

===== STEP 3: Detecting outliers =====
Time taken in STEP 3:  0.009708404541015625
Outlier CRs:  ['CR-1', 'CR-2', 'CR-4', 'CR-15']
Valid CRs:  ['CR-3', 'CR-5', 'CR-6', 'CR-7', 'CR-8', 'CR-9', 'CR-10', 'CR-11', 'CR-12', 'CR-13', 'CR-14', 'CR-16']
Threshold: 0.029690608333333333
Mean distances per CR (index i = CR-(i+1)): [0.05658, 0.05762, 0.02675, 0.02981, 0.0279, 0.02736, 0.0268, 0.02685, 0.0276, 0.02949, 0.02608, 0.02732, 0.02855, 0.02857, 0.02981, 0.02912]


### Custom dist + segment-wise optimization: 

In [141]:
def compute_distance_matrix_in_segments_custom_dist(
    rounds,
    curves,
    num_segments=6,
    segment_indices=[0, 3, 5]
):
    """
    Computes DTW distance matrix using segment-wise optimization.

    Strategy:
    - Split each curve into segments
    - Compute DTW only on selected segments
    - Average segment-wise DTW costs

    Returns:
        distance_df (pd.DataFrame)
    """
    start_time = time.time()
    print("Computing segment-wise DTW distance matrix...")

    # -------------------------------
    # Step 1: Precompute segments
    # -------------------------------
    segmented_curves = [extract_segments(c, num_segments, segment_indices) for c in curves]

    # -------------------------------
    # Step 2: Distance computation
    # -------------------------------
    N = len(segmented_curves)
    distance_matrix = np.zeros((N, N))

    for i in range(N):
        for j in range(i, N):

            if i == j:
                dist = 0

            else:
                segs_i = segmented_curves[i]
                segs_j = segmented_curves[j]

                dist_sum = 0
                valid_seg_count = 0

                for seg_i, seg_j in zip(segs_i, segs_j):
                    dist_ij = dtw.distance(np.array(seg_i, dtype=np.double), np.array(seg_j, dtype=np.double), inner_dist='euclidean', use_c=True)

                    dist_sum += dist_ij
                    valid_seg_count += 1

                # Avoid division by zero
                if valid_seg_count == 0:
                    dist = 0
                else:
                    dist = dist_sum 

            distance_matrix[i, j] = dist
            distance_matrix[j, i] = dist
            print(f"CR {i+1}-{j+1}: {dist}")
    
    elapsed_time = time.time() - start_time 
    print("Time taken: ", elapsed_time)

    return pd.DataFrame(distance_matrix, index=rounds, columns=rounds)

In [142]:
unique_crs = data['cr'].unique()
print(type(unique_crs))

distance_df = compute_distance_matrix_in_segments_custom_dist(unique_crs, cr_load_curves)

<class 'pandas.arrays.StringArray'>
Computing segment-wise DTW distance matrix...
CR 1-1: 0
CR 1-2: 1.144927536231884
CR 1-3: 18.65217391304349
CR 1-4: 25.028985507246375
CR 1-5: 22.811594202898565
CR 1-6: 19.173913043478272
CR 1-7: 23.18840579710146
CR 1-8: 17.608695652173928
CR 1-9: 19.304347826086968
CR 1-10: 24.39130434782608
CR 1-11: 22.956521739130444
CR 1-12: 19.173913043478272
CR 1-13: 23.66666666666668
CR 1-14: 17.826086956521756
CR 1-15: 23.666666666666675
CR 1-16: 19.855072463768135
CR 2-2: 0
CR 2-3: 19.710144927536238
CR 2-4: 26.05797101449275
CR 2-5: 23.840579710144944
CR 2-6: 20.13043478260871
CR 2-7: 23.826086956521753
CR 2-8: 18.02898550724639
CR 2-9: 20.30434782608696
CR 2-10: 25.376811594202895
CR 2-11: 23.985507246376827
CR 2-12: 20.13043478260871
CR 2-13: 24.362318840579718
CR 2-14: 18.246376811594214
CR 2-15: 24.362318840579718
CR 2-16: 20.69565217391306
CR 3-3: 0
CR 3-4: 11.739130434782611
CR 3-5: 14.188405797101467
CR 3-6: 17.913043478260875
CR 3-7: 18.0869565217

In [143]:
_, outlier_rounds, _ = detect_outliers(distance_df, 0.05)

===== STEP 3: Detecting outliers =====
Time taken in STEP 3:  0.010039567947387695
Outlier CRs:  ['CR-1', 'CR-2', 'CR-4', 'CR-10']
Valid CRs:  ['CR-3', 'CR-5', 'CR-6', 'CR-7', 'CR-8', 'CR-9', 'CR-11', 'CR-12', 'CR-13', 'CR-14', 'CR-15', 'CR-16']
Threshold: 15.898067632850246
Mean distances per CR (index i = CR-(i+1)): [19.89662, 20.68019, 14.96232, 18.2744, 15.14396, 12.35556, 12.24155, 11.46957, 14.38357, 16.48599, 15.47536, 12.35556, 12.50918, 11.03188, 13.03285, 12.08599]
